# 01. Dashboard bring-up — `arcui`

`arcui` is the **observability dashboard** for an Arc agent fleet. It is a Starlette app with a WebSocket transport that receives live telemetry from running agents and renders a real-time UI for monitoring LLM calls, tool invocations, costs, and audit events.

This notebook walks the *bring-up* surface — the part you touch when you bolt the dashboard onto an existing process. It does **not** cover live attach, the WS protocol, or the routes themselves. Those are the subject of `02-live-telemetry-attach.ipynb`.

**You will learn:**

- What `arcui` is and where it sits in the Arc dependency graph
- How to build a Starlette app with `create_app()`
- The three-token auth model (operator / viewer / agent) and why it exists
- How to run the server with `serve()` — and how to drive it in-process via `TestClient`
- The high-level routing structure and which compliance controls each surface satisfies
- Operational notes for binding, TLS, and reverse proxies in real deployments

Every cell runs **without an API key** and without binding a real port. Where binding a port is the right thing to demonstrate, we use a `subprocess` with a hard timeout so the notebook always terminates.

## 1. Setup

The standard Arc walkthrough boilerplate. It locates the repo root and adds every `packages/<pkg>/src` (and `tests/`) directory to `sys.path` so the notebook can import `arcui`, `arcllm`, and `arctrust` without a `pip install -e .` cycle.

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

Confirm the package imports cleanly. If this fails, the boilerplate above did not find the Arc repo root.

In [ ]:
import arcui
from arcui import create_app, serve, attach_llm

print(f"arcui version: {arcui.__version__}")
print(f"public surface: {arcui.__all__}")

`arcui.__all__` is the entire stable public API of the package. Three names — `create_app`, `serve`, `attach_llm` — and that is the whole bringup story. We will spend the next sections unpacking each one.

## 2. What `arcui` is

Three layers, top to bottom:

1. **Browser dashboard** — a single-page app served at `/` that renders the live event stream, fleet roster, audit trail, cost totals, and LLM telemetry.
2. **REST + WebSocket server** — a Starlette app that accepts `wss://.../ws` from browser viewers and `wss://.../api/agent/connect` from running agents, plus `GET /api/...` queries for traces, stats, and configuration.
3. **In-memory plumbing** — a `ConnectionManager`, `RollingAggregator`, `EventBuffer`, `SubscriptionManager`, and `AgentRegistry` that fan events from agents to subscribed browser tabs.

Where it sits in the Arc graph:

In [ ]:
# arcui is downstream of arcllm (for JSONLTraceStore) and arctrust
# (for UIBridgeSink). arcagent and arccli depend on arcui — agents
# stream into it; the CLI starts it with `arc ui start`.
import importlib

deps = {}
for name in ("arcui", "arcllm", "arctrust", "arcgateway"):
    try:
        mod = importlib.import_module(name)
        deps[name] = getattr(mod, "__version__", "<no version>")
    except ImportError as exc:
        deps[name] = f"<missing: {exc}>"

for k, v in deps.items():
    print(f"  {k:12s} {v}")

Notice that arcui depends on `arcgateway` for fleet plumbing (team roster, file-event bus, fs watcher). Bringing up the dashboard alongside a real agent fleet means optionally passing a `team_root` and a `gateway_config` to `create_app()` — we will see that pattern in §3.

In [ ]:
# The package layout, summarized.
import pkgutil
import arcui as arcui_pkg

modules = sorted(info.name for info in pkgutil.iter_modules(arcui_pkg.__path__))
for m in modules:
    print(f"  arcui.{m}")

Highlights:

- `server.py` — the app factory and uvicorn runner (this notebook)
- `auth.py` — the three-token auth middleware
- `routes/` — every HTTP route the dashboard exposes (covered in §6)
- `connection.py`, `event_buffer.py`, `subscription.py`, `registry.py` — the fan-out plumbing
- `bridge.py`, `reporter.py` — the agent → dashboard bridge (covered in `02-live-telemetry-attach.ipynb`)

## 3. `create_app` — building the Starlette app

`create_app()` is the factory. It returns a fully wired Starlette application that you can hand to **any** ASGI server (uvicorn, hypercorn, daphne) or drive in-process from a test harness.

Calling it with no arguments gives you a self-contained dashboard with auto-generated tokens, no agent fleet, and no trace store:

In [ ]:
from arcui import create_app

app = create_app()

print(type(app).__name__)
print("routes registered:", len(app.routes))

Every piece of plumbing is on `app.state` — that is the contract. Routes do not import the registry, the aggregator, or the trace store directly. They reach into `app.state` so the wiring stays in one place and tests can swap pieces out.

In [ ]:
# Inspect the wired-up state. This is the contract every route relies on.
state_keys = [
    k
    for k in dir(app.state)
    if not k.startswith("_") and not callable(getattr(app.state, k, None))
]
for k in sorted(state_keys):
    val = getattr(app.state, k)
    print(f"  app.state.{k:24s} -> {type(val).__name__}")

Several of these matter for bring-up:

- `agent_registry` — tracks online agents (max-cap configurable)
- `event_buffer` — bounded ring buffer of recent events
- `aggregator` — sliding-window cost / latency / token rollups
- `connection_manager` — viewer WebSocket sessions
- `subscription_manager` — topic-based event subscription
- `audit` — the `UIAuditLogger` that emits `ui.session_start` and friends
- `session_tracker` — at-most-once session-start emission per (token, address)
- `auth_config` — the resolved three-token bundle

All of these are *optional* to interact with directly. The point of `create_app()` is that the wiring is correct without you touching any of them.

In [ ]:
# The factory accepts a handful of keyword arguments that control how
# the dashboard composes with the rest of Arc:
import inspect

from arcui.server import create_app as _factory

sig = inspect.signature(_factory)
for name, param in sig.parameters.items():
    print(f"  {name:20s} default={param.default!r}")

Each parameter has a clear job:

- `auth_config` — supply your own `AuthConfig`; otherwise tokens are auto-generated.
- `trace_store` — an `arcllm.JSONLTraceStore` so `/api/traces` and warm-start replay work.
- `config_controller` — an `arcllm.ConfigController` so the Settings page can mutate runtime config.
- `agent_info` — metadata about *this* agent (when `arcui` is embedded in `arc agent serve`).
- `max_agents` — connection cap for the registry (default `100`).
- `team_root` — path to a `team/` directory; enables the fleet roster endpoints.
- `gateway_config` — when set, composes an in-process `arcgateway` runtime.

In [ ]:
# Build the app with explicit knobs to see how it changes.
from unittest.mock import MagicMock

fake_trace_store = MagicMock(name="fake_trace_store")
fake_config_ctrl = MagicMock(name="fake_config_controller")
agent_info = {
    "name": "demo-agent",
    "did": "did:arc:local:executor/abc123",
    "model": "anthropic/claude-sonnet-4-6",
    "provider": "anthropic",
}

app2 = create_app(
    trace_store=fake_trace_store,
    config_controller=fake_config_ctrl,
    agent_info=agent_info,
    max_agents=25,
)

print("trace_store:    ", app2.state.trace_store)
print("config_ctrl:    ", app2.state.config_controller)
print("agent_info:     ", app2.state.agent_info)
print("max_agents cap: ", app2.state.agent_registry.max_agents)

`MagicMock` stand-ins are fine here because `create_app()` is a *wiring* function — it stores the references, it does not call into them. The trace store is consulted later, by the `traces_routes` handlers, which we exercise via `TestClient` in §5.

## 4. The three-token auth model

Most observability dashboards ship with a single shared password. That fails in two predictable ways:

1. The read-only viewer who sees a chart can also kill an agent or push admin commands.
2. The credential the *agent* uses to push events is the same one a *human* uses to read them.

`arcui` solves both with three tokens:

In [ ]:
from arcui.auth import AuthConfig

# Auto-generate. Each token is 32 random bytes hex-encoded (64 chars).
cfg = AuthConfig()

print("viewer_token len  :", len(cfg.viewer_token))
print("operator_token len:", len(cfg.operator_token))
print("agent_token len   :", len(cfg.agent_token))
print("all distinct      :", len({cfg.viewer_token, cfg.operator_token, cfg.agent_token}) == 3)

Each token maps to one role. The mapping is enforced by `AuthMiddleware`, which sits in front of every `/api/*` route:

| Token | Role | Allowed |
|---|---|---|
| viewer | read-only | dashboard, traces, stats, `arc ui tail` |
| operator | read + control | viewer + approve pairings, kill sessions |
| agent | push-only | `/api/agent/connect` WebSocket; **rejected** on REST routes |

Validation is `hmac.compare_digest` so a network-timing attack cannot enumerate the token by length-of-prefix.

In [ ]:
# Supply your own tokens (e.g. from a secrets manager) instead of
# auto-generating. The constructor accepts a plain dict so an Ansible
# playbook or systemd unit can hand over secrets without importing
# anything Arc-specific.
supplied = AuthConfig(
    {
        "viewer_token": "viewer-demo-token",
        "operator_token": "operator-demo-token",
        "agent_token": "agent-demo-token",
    }
)

print("viewer   ->", supplied.validate_token("viewer-demo-token"))
print("operator ->", supplied.validate_token("operator-demo-token"))
print("agent    ->", supplied.validate_token("agent-demo-token"))
print("unknown  ->", supplied.validate_token("not-a-real-token"))

Pass the resolved `AuthConfig` to `create_app(auth_config=...)`. From that point on, the middleware enforces the policy on every request.

In [ ]:
# Build an app with our known tokens and inspect the middleware layer.
from arcui.auth import AuthMiddleware

app_with_auth = create_app(auth_config=supplied)

middleware_classes = [m.cls.__name__ for m in app_with_auth.user_middleware]
print("middleware stack:", middleware_classes)
assert "AuthMiddleware" in middleware_classes

**On disk: `~/.arcagent/ui-token`.** When `arc ui start` boots, the agent token (and only the agent token) is persisted with `0600` permissions. Any agent process on the host can find the dashboard URL + token without env vars or service discovery. The viewer and operator tokens never touch disk — they are printed once and meant for human eyes.

## 5. Bringing up the server — `serve()` and `TestClient`

`serve()` is the one-liner. It:

1. Calls `create_app()` with your kwargs.
2. Optionally calls `attach_llm(app, llm)` if you passed an `arcllm` model.
3. Warm-starts the rolling aggregator from a trace store, if one is supplied.
4. Hands the app to `uvicorn.run(host, port)`.

Step 4 is a *blocking* call. In a notebook we never want to block — so we drive the app **in-process** via `starlette.testclient.TestClient`, which goes through the full ASGI stack but never opens a real socket.

In [ ]:
import inspect

print(inspect.signature(serve))
print()
print(inspect.getdoc(serve))

Same kwargs as `create_app()`, plus `host`, `port`, and an `llm`. The defaults are `127.0.0.1:8420` — local-by-default, never `0.0.0.0`. We will come back to that choice in §8.

In [ ]:
# Drive the app in-process with TestClient. No socket is bound.
from starlette.testclient import TestClient

auth = AuthConfig(
    {
        "viewer_token": "viewer-tok",
        "operator_token": "operator-tok",
        "agent_token": "agent-tok",
    }
)
tc_app = create_app(auth_config=auth)
client = TestClient(tc_app)

# Liveness probe — exempt from auth.
resp = client.get("/api/health")
print("GET /api/health:", resp.status_code, resp.json())

Health is exempt by design — Kubernetes liveness probes must not need credentials. Every other `/api/*` route does require a token.

In [ ]:
# Without a token: 401.
resp = client.get("/api/info")
print("no token:    ", resp.status_code, resp.json())

# Wrong token: 401.
resp = client.get("/api/info", headers={"Authorization": "Bearer not-real"})
print("bad token:   ", resp.status_code, resp.json())

# Viewer token: 200.
resp = client.get("/api/info", headers={"Authorization": "Bearer viewer-tok"})
print("viewer ok:   ", resp.status_code, resp.json())

Now the agent-token-on-REST rule (ASI-03 — privilege abuse). An agent token is for the `/api/agent/connect` WebSocket *only*; using it to read traces or stats is a 403, not a 401, because the credential is valid but the role is wrong.

In [ ]:
resp = client.get("/api/info", headers={"Authorization": "Bearer agent-tok"})
print("agent on REST:", resp.status_code, resp.json())

Now exercise the `agent_info` surface — the `/api/info` endpoint reflects whatever metadata you passed at construction. This is what the dashboard reads to render the agent's name and DID in the title bar:

In [ ]:
info_app = create_app(
    auth_config=auth,
    agent_info={
        "name": "demo-agent",
        "did": "did:arc:local:executor/abc123",
        "model": "anthropic/claude-sonnet-4-6",
    },
)
info_client = TestClient(info_app)

resp = info_client.get("/api/info", headers={"Authorization": "Bearer viewer-tok"})
print(resp.status_code)
for k, v in resp.json().items():
    print(f"  {k:8s} {v}")

**The subprocess pattern.** When you genuinely need to verify a real port-bound bringup (smoke-test, packaging release gate), run `serve()` in a subprocess with a hard timeout. The cell below documents the shape; we leave it commented so the notebook never actually binds a port:

In [ ]:
# Documented pattern — DO NOT uncomment in this notebook.
#
# import subprocess, sys, time
#
# proc = subprocess.Popen(
#     [sys.executable, "-c",
#      "from arcui import serve; serve(host='127.0.0.1', port=18420)"],
#     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
# )
# try:
#     time.sleep(1.0)            # wait for uvicorn boot
#     # ... hit http://127.0.0.1:18420/api/health with httpx ...
# finally:
#     proc.terminate()
#     proc.wait(timeout=5)
print("subprocess pattern documented — TestClient is preferred for notebook tests.")

`TestClient` is preferred because it covers everything except `uvicorn.run()` itself — every route, every middleware, every lifespan startup hook fires. The subprocess is only useful when the *exact* `uvicorn` binding behavior is what you are testing.

## 6. Routing structure — what the dashboard exposes

`create_app()` registers routes in a fixed order. We can enumerate them off the live app:

In [ ]:
# Pull the route table off the live app.
from starlette.routing import Mount, Route, WebSocketRoute

summary = []
for r in app.routes:
    if isinstance(r, Route):
        methods = ",".join(sorted(r.methods or set())) if r.methods else "-"
        summary.append(("HTTP", r.path, methods))
    elif isinstance(r, WebSocketRoute):
        summary.append(("WS", r.path, "-"))
    elif isinstance(r, Mount):
        summary.append(("MOUNT", r.path, "-"))

print(f"Total routes: {len(summary)}\n")
for kind, path, methods in summary[:25]:
    print(f"  {kind:6s} {path:48s} {methods}")
print("  ...")

Grouped by purpose, the surface is small and predictable:

| Group | Examples | Source module |
|---|---|---|
| Liveness / metadata | `/api/health`, `/api/info` | `server.py` |
| Dashboard SPA | `/`, `/sw.js`, `/assets/*` | `server.py` |
| Trace queries | `/api/traces`, `/api/traces/{id}` | `routes.traces` |
| Stats & cost | `/api/stats/*`, `/api/cost-efficiency` | `routes.stats`, `routes.cost_efficiency` |
| Config | `/api/config`, `/api/arcllm-config` | `routes.config`, `routes.arcllm_config` |
| Agent fleet | `/api/agents`, `/api/agents/{id}`, `/api/team/*` | `routes.agents`, `routes.team_pages` |
| Schedules | `/api/schedules` | `routes.schedules` |
| Knowledge | `/api/agents/{id}/knowledge/*` | `routes.knowledge` |
| Export | `/api/export/*` | `routes.export` |
| WebSockets | `/ws`, `/api/agent/connect`, `/api/chat`, `/api/dashboard` | `routes.ws`, `routes.agent_ws`, `routes.chat_ws`, `routes.dashboard_ws` |

In [ ]:
# Quick sanity-check that the well-known REST endpoints are all wired.
expected = [
    "/",
    "/api/health",
    "/api/info",
    "/api/traces",
    "/api/agents",
    "/api/agent/connect",
]
paths = {r.path for r in app.routes if hasattr(r, "path")}
for p in expected:
    print(f"  {p:24s} {'OK' if p in paths else 'MISSING'}")

`/api/agent/connect` is a WebSocket route — its presence here confirms the agent-push channel is registered alongside the human-facing routes. The browser hits `/ws`; the agent hits `/api/agent/connect`. They never share a path, never share a token.

In [ ]:
# Example: list traces with no store configured. Returns an empty page.
resp = client.get("/api/traces", headers={"Authorization": "Bearer viewer-tok"})
print(resp.status_code)
print(resp.json())

With a `trace_store` wired in, the same endpoint returns real records. We will exercise that path properly in `02-live-telemetry-attach.ipynb`.

In [ ]:
# The dashboard SPA itself. Index serves the cached HTML; static assets
# are mounted under /assets so the browser fetches JS/CSS without
# touching the auth middleware.
resp = client.get("/")
ct = resp.headers.get("content-type", "")
cc = resp.headers.get("cache-control", "")
print(f"GET / status={resp.status_code} content-type={ct!r} cache-control={cc!r}")

`Cache-Control: no-store` on the SPA shell means a stale tab cannot survive an `arc ui start` restart with old asset references — important because every restart issues fresh tokens, and a stale tab serving cached JS would call dead endpoints.

## 7. Compliance hooks — what auditors care about

The README has the canonical mapping, but two things are worth surfacing here because they show up in the wiring you just built:

**NIST 800-53:**

| Control | Where it lives in `create_app()` |
|---|---|
| AU-2 (audit events) | `app.state.audit` (`UIAuditLogger`) — emits `ui.session_start`, `ui.token_invalid`, etc. |
| AU-9 (read-only audit display) | The dashboard reads `app.state.audit_buffer`; there is no write path from the UI back to the trace store. |
| AU-11 (retention) | `trace_store=` (an `arcllm.JSONLTraceStore`) — daily file rotation, hash-chained records. |
| SI-4 (continuous monitoring) | The WebSocket transport pushes events sub-second; `RollingAggregator` keeps a 1h / 24h / 7d window. |
| SC-13 (cryptographic role separation) | `AuthConfig` — three tokens, one role each, `hmac.compare_digest` validation. |

**OWASP Agentic 2026:**

| Risk | Mitigation in `arcui` |
|---|---|
| ASI-03 (Identity & Privilege Abuse) | Agent tokens 403 on REST; viewer tokens cannot push events. |
| ASI-09 (Trust Exploitation) | Every agent action surfaces with `agent_id` + `did` attribution; humans can see impersonation in real time. |
| ASI-10 (Rogue Agents) | The dashboard *is* the behavioral monitoring surface; anomalies show up the moment they emit. |

In [ ]:
# Confirm the audit logger is wired and emitting structured events.
import logging

logger = app.state.audit
print("logger type:", type(logger).__name__)

# UIAuditLogger writes structured records to the configured logger.
# Plumbing-level smoke: the logger object exists, has the expected
# audit_event method.
assert hasattr(logger, "audit_event"), "audit logger must expose audit_event()"
print("audit_event method present: OK")

The `audit_event()` method is what the auth middleware calls when it hands out a fresh `session_id` to a new (token, remote_addr) pair. SR-3 / SPEC-025 §FR-7 mandate five fields per event — `session_id`, `uid`, `username`, `remote_addr`, `auth_method` — encoded as a Pydantic model so dropping a field is a type error, not a silent audit gap.

In [ ]:
# The session tracker is the at-most-once gate.
from arcui.auth import SessionTracker

tracker = app.state.session_tracker
print("tracker type:", type(tracker).__name__)
print("is SessionTracker:", isinstance(tracker, SessionTracker))

Two interesting properties of `SessionTracker`:

1. **Token hashing.** Tokens are SHA-256 hashed before storage so a memory dump or audit log of session ids cannot be reversed to the bearer token (SR-2).
2. **LRU bounding.** Both internal stores are bounded LRUs; on overflow the oldest entry is evicted, which re-emits `ui.session_start` for a long-idle client when it returns. That re-emission is the *correct* audit behavior, not a bug — it preserves the chain across long quiet periods.

In [ ]:
# The bounds are env-configurable for federal tuning.
import os

for var in ("ARCUI_MAX_SESSIONS", "ARCUI_MAX_BOOTSTRAP_MARKERS"):
    print(f"  {var:32s} -> {os.environ.get(var, '(unset, default)')}")

The defaults — 10 000 sessions, 1 000 bootstrap markers — cap memory at well under 5 MB and are safe for a long-running federal deployment without explicit tuning.

## 8. Operational notes — TLS, binding, reverse proxy, port choice

`serve()` is *intentionally* opinionated:

- **Bind defaults to `127.0.0.1`.** Not `0.0.0.0`. The dashboard is local-by-default. To bind all interfaces you must pass `host='0.0.0.0'` explicitly — and you should put it behind mTLS or a reverse proxy when you do.
- **No TLS in `serve()` itself.** `serve()` runs plain HTTP. For TLS, terminate at a reverse proxy (Caddy, nginx, Traefik) or hand the app to a uvicorn invocation with `ssl_keyfile=` / `ssl_certfile=`.
- **Port `8420` by default.** Why 8420: outside common reserved ranges, easy to remember, and matches the rest of the Arc tooling.
- **Single-process.** `uvicorn.run(app, ...)` runs one worker. To horizontally scale, front the dashboard with a process manager that points many workers at a shared trace dir.

In [ ]:
# Verify the defaults match the README and the docstring.
import inspect

params = inspect.signature(serve).parameters
print("host default:", params["host"].default)
print("port default:", params["port"].default)

# Local-by-default invariant.
assert params["host"].default == "127.0.0.1", "federal-first: bind localhost by default"
assert params["port"].default == 8420, "port choice changed — update README"

Keeping `serve()` opinionated about binding is a federal-mode invariant. It is the difference between a developer accidentally exposing live agent telemetry on a lab subnet vs. forcing them to write `--host 0.0.0.0` deliberately and notice the choice.

In [ ]:
# Reverse-proxy front: the app is plain ASGI; pass it to uvicorn yourself
# when you need control over workers, ssl, or proxy headers.
#
# Example invocation (do NOT execute in the notebook):
#
#   uvicorn arcui.server:create_app --factory \
#       --host 0.0.0.0 --port 8420 \
#       --workers 1 \
#       --proxy-headers --forwarded-allow-ips '127.0.0.1' \
#       --log-level info
#
# Then in nginx / Caddy:
#
#   reverse_proxy 127.0.0.1:8420 {
#       header_up X-Forwarded-Proto {scheme}
#       header_up Host             {host}
#   }
print("see comment for the canonical reverse-proxy invocation")

**Why `--proxy-headers` matters.** `arcui` reads `request.client.host` to attribute audit events. Without `--proxy-headers` the audit log shows the proxy's IP for every viewer. With it, uvicorn unwraps `X-Forwarded-For` and you see the real client.

**Why `--workers 1`.** The aggregator and event buffer are *in-process* state. Two uvicorn workers would fragment the rolling stats across processes. To scale horizontally, run multiple `arcui` instances behind a load balancer with sticky sessions on the WebSocket endpoint and a shared trace store on disk.

In [ ]:
# Sanity-check that the package re-exports stay tight.
from arcui import __all__ as exported

expected = {"__version__", "create_app", "serve", "attach_llm"}
actual = set(exported)
missing = expected - actual
extra = actual - expected
print("missing:", missing or "none")
print("extra:  ", extra or "none")
assert not missing, f"public surface lost a name: {missing}"
assert not extra, f"public surface grew unexpectedly: {extra}"

If this assertion ever fires, the public surface has drifted and either `__init__.py` or this notebook needs updating. The point of pinning the surface in a smoke test is that drift cannot land silently.

In [ ]:
# Tear down. TestClient holds an internal task group; closing it is hygienic
# even though the next garbage-collection pass would do it for us.
client.close()
info_client.close()
print("clients closed; notebook is ready for the next walkthrough.")

## 9. Summary

We brought the dashboard up three different ways and never opened a real port:

- **`create_app()`** is the Starlette factory. Three optional kwargs of note: `auth_config`, `trace_store`, `agent_info`. Everything is wired onto `app.state` so routes never reach for module-level singletons.
- **`serve()`** is the one-liner that bundles `create_app()` + optional `attach_llm()` + `uvicorn.run()`. Local-by-default (`127.0.0.1:8420`), no TLS — terminate TLS at a reverse proxy.
- **Three tokens, three roles.** `viewer` reads, `operator` reads + controls, `agent` pushes events. `hmac.compare_digest` validation; agent tokens 403 on REST routes (ASI-03).
- **`TestClient` is the right notebook tool.** It exercises the full ASGI stack — middleware, lifespan, routes — without binding a socket. Subprocess + timeout is documented but reserved for packaging-level smoke tests.
- **Routing surface is small and stable.** `/api/health`, `/api/info`, traces, stats, agents, schedules, knowledge, export, plus four WebSocket endpoints (`/ws`, `/api/agent/connect`, `/api/chat`, `/api/dashboard`).
- **Compliance hooks live in the wiring.** `UIAuditLogger`, `SessionTracker` with hashed tokens and bounded LRUs, mandatory five-field session-start events. NIST AU-2/9/11, SI-4, SC-13. OWASP Agentic ASI-03 / ASI-09 / ASI-10.
- **Operational invariants.** Bind localhost by default, single uvicorn worker, terminate TLS upstream, rely on `--proxy-headers` for accurate audit attribution.

**Public API exercised in this notebook:** `arcui.__version__`, `arcui.create_app`, `arcui.serve`, `arcui.attach_llm` (referenced; full coverage in 02), `arcui.auth.AuthConfig`, `arcui.auth.AuthMiddleware`, `arcui.auth.SessionTracker`.

**Next: see `02-live-telemetry-attach.ipynb`** — wires an `arcllm` model into the running app via `attach_llm`, walks the WebSocket protocol the browser uses, and exercises live trace + stat queries against a real `JSONLTraceStore`.